# Exploratory Data Analysis

Thin by design: all reusable logic lives in `src/`. This notebook only loads data through `src.data.load` and visualizes the panel structure and target distribution -- it does not implement feature engineering or modeling (see `src/training/train.py` for that).

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.load import load_raw_train

df = load_raw_train("../data/raw/train.csv")
df.shape

## Panel structure

This dataset is a perfectly balanced weekly panel: every (store, SKU) combination has an observation for every week in range.

In [ ]:
print("stores:", df["store_id"].nunique())
print("skus:", df["sku_id"].nunique())
print("store-sku combos:", df.groupby(["store_id", "sku_id"]).ngroups)
print("weeks:", df["week"].nunique(), df["week"].min().date(), "->", df["week"].max().date())
df.groupby(["store_id", "sku_id"]).size().describe()

In [ ]:
weeks = sorted(df["week"].unique())
gaps = pd.Series(weeks).diff().dropna()
gaps.value_counts()

Note the single 8-day gap (2012-02-27 -> 2012-03-06): a genuine anomaly in the source calendar, not missing data (`tests/test_dataset_invariants.py` and `src/data/validate.py::validate_panel_completeness` both surface this).

## Target distribution

`units_sold` is right-skewed count data -- the training pipeline fits a `log1p`-transformed target for this reason (see `configs/train.yaml: features.log_transform_target`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["units_sold"], bins=60, color="#2E5EAA")
axes[0].set_title("units_sold")
axes[1].hist(np.log1p(df["units_sold"]), bins=60, color="#2E5EAA")
axes[1].set_title("log1p(units_sold)")
fig.tight_layout()
df["units_sold"].describe()

## Promotion effect

Featured/display placement is associated with materially higher demand -- both are kept as model features.

In [ ]:
df.groupby("is_featured_sku")["units_sold"].mean(), df.groupby("is_display_sku")["units_sold"].mean()